# Tính năng 4 – Trang **Đánh giá mô hình** (phần 1: huấn luyện và so sánh)

Notebook trả lời: mô hình nào được chọn và vì sao, đoán sai bao nhiêu, đoán sai ở đâu, yếu tố nào quyết định lương, và kỹ năng nào đáng học. Phần độ tin cậy và chuẩn đưa vào sử dụng nằm ở notebook tính năng 5.

> **Từ cần biết:** *tập học (train)* là phần dữ liệu mô hình được xem để học; *tập chọn (valid)* dùng để so và chọn mô hình; *tập kiểm tra (test)* chỉ mở một lần ở cuối để chấm điểm thật. *Trung vị* là giá trị đứng giữa khi xếp từ thấp đến cao. *MAE* là trung bình mô hình đoán lệch bao nhiêu triệu đồng mỗi tin.

Mã nguồn: `src/career_market/model/estimators.py`, `src/career_market/model/training.py`, `pages/evaluation.py`.

In [ ]:
# Thiết lập: dùng đúng thư viện của ứng dụng (job_market/src), dữ liệu và mô hình đã lưu.
import sys
import warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "career_market").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from career_market.ui import charts  # thư viện biểu đồ chọn backend "Agg" (vẽ cho web)…
%matplotlib inline
# …nên bật lại chế độ inline để biểu đồ hiện trong notebook.
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

In [ ]:
from career_market.data.dataset import load_dataset
from career_market.model.estimators import MODEL_LABELS, build_model, feature_frame
from career_market.model.training import load_report, load_skill_value, scores
from career_market.paths import artifacts_dir

frame = load_dataset()
report = load_report()
parts = {s: frame[frame["split"] == s] for s in ["train", "valid", "test"]}
print({s: len(p) for s, p in parts.items()})

## 1. Mô hình được xem gì – và cố ý **không** được xem gì
Mô hình được xem: vai trò, khu vực, cấp bậc, **trang đăng tin, hình thức làm việc, loại công ty**, số năm kinh nghiệm, năm của tin, 4 dấu hiệu từ chức danh (chuyên gia, trưởng…) và danh sách kỹ năng.

Mô hình **không** được xem tên công ty, tiêu đề gốc hay chuỗi lương. Nếu biết tên công ty, mô hình sẽ học kiểu "công ty X trả cao", và điều đó vô dụng khi người dùng hỏi về một công ty mới.

In [ ]:
X = feature_frame(parts["train"])
print("Số cột đầu vào trước khi mã hóa:", X.shape[1])
X.iloc[:3, :12]

## 2. Bốn ứng viên – như bốn cách con người ước lượng lương

| Mô hình | Giống như ngoài đời | Điểm yếu |
|---|---|---|
| **Mốc trung vị** | Ai hỏi cũng trả lời "khoảng 25 triệu" | Chỉ là mức tối thiểu để so |
| **Ridge** | Một bảng cộng điểm: Backend +2, mỗi năm kinh nghiệm +3, biết AWS +1 | Không hiểu "còn tùy hoàn cảnh" |
| **Random Forest** | Hỏi 300 chuyên viên, mỗi người xem một phần hồ sơ, rồi lấy trung bình | Từng người dễ "học thuộc" ví dụ đã thấy |
| **Gradient Boosting** | Một hội đồng làm nối tiếp: người sau chỉ sửa chỗ người trước đoán sai | Khó giải thích hơn Ridge |

**Ba nguyên tắc khi huấn luyện:**
- Học trên *tỉ lệ* chứ không phải số tuyệt đối: lệch 5 triệu với người lương 10 triệu nghiêm trọng hơn nhiều so với người lương 100 triệu.
- Chọn mô hình bằng sai số trên **tập chọn**, và chốt cách chọn từ trước.
- **Tập kiểm tra cuối** chỉ mở một lần để chấm, không dùng để chọn.

In [ ]:
y = parts["train"]["salary"]
print(f"Skew của lương: {y.skew():.2f} → sau logarit: {np.log(y).skew():.2f}")
rows = [{"Mô hình": MODEL_LABELS[n], "Siêu tham số": t["params"], "MAE valid": t["valid_MAE"]} for n, trials in report["tuning"].items() for t in trials]
tuning = pd.DataFrame(rows)
tuning.assign(**{"Chọn": np.where(tuning.groupby("Mô hình")["MAE valid"].transform("min") == tuning["MAE valid"], "✓", "")})

## 3. Quy tắc đời thường: thêm kinh nghiệm hay kỹ năng không thể làm giảm lương

**Vấn đề:** thêm kỹ năng Java vào hồ sơ Backend 5 năm từng làm lương dự đoán *giảm* từ 47,7 xuống 42,3 triệu. Mô hình không bị lỗi lập trình mà học đúng dữ liệu: cột kỹ năng ghi những gì **tin tuyển dụng yêu cầu**, và tin liệt kê nhiều kỹ năng thường là tin tuyển người mới hoặc tuyển hàng loạt, lương thấp. Bảng dưới kiểm tra điều đó trên dữ liệu hiện tại:

In [ ]:
backend = frame[frame["role"] == "Backend Developer"]
bins = pd.cut(backend["skill_count"], [-1, 0, 2, 4, 6, 30], labels=["0", "1–2", "3–4", "5–6", "7+"])
backend.groupby(bins, observed=False)["salary"].agg(["size", "median"]).rename(columns={"size": "Số tin", "median": "Trung vị lương"}).T

**Cách xử lý:** bỏ hẳn thông tin "số lượng kỹ năng", và buộc cả ba mô hình tuân theo quy tắc "thêm kỹ năng hoặc kinh nghiệm thì lương không giảm" (mỗi mô hình một cách kỹ thuật riêng). Kiểm tra trên mô hình đã lưu:

In [ ]:
from career_market.model.advisor import predict_profile

base = {"role": "Backend Developer", "region": "Hà Nội", "experience_years": 5}
pd.DataFrame([{"Mô hình": MODEL_LABELS[n],
               "Không kỹ năng": predict_profile({**base, "skills": []}, n)["predicted"],
               "+ Java": predict_profile({**base, "skills": ["Java"]}, n)["predicted"],
               "+ Java, Spring Boot, Docker, AWS": predict_profile({**base, "skills": ["Java", "Spring Boot", "Docker", "AWS"]}, n)["predicted"]}
              for n in ["GradientBoosting", "RandomForest", "Ridge"]]).set_index("Mô hình")

## 4. Chạy lại để kiểm chứng: huấn luyện lại Gradient Boosting với thiết lập đã chọn

In [ ]:
params = report["models"]["GradientBoosting"]["params"]
model = build_model("GradientBoosting", **params).fit(feature_frame(parts["train"]), parts["train"]["salary"])
again = {s: scores(parts[s]["salary"], model.predict(feature_frame(parts[s])))["MAE"] for s in ["train", "valid", "test"]}
pd.DataFrame({"Fit lại trong notebook": again, "Trong metrics.json": {s: report["models"]["GradientBoosting"][s]["MAE"] for s in again}})

## 5. Kết quả trên ba tập dữ liệu

In [ ]:
rows = [{"Mô hình": v["label"], "MAE train": v["train"]["MAE"], "MAE valid": v["valid"]["MAE"], "MAE test": v["test"]["MAE"],
         "RMSE test": v["test"]["RMSE"], "R² test": v["test"]["R2"], "MAPE test %": v["test"]["MAPE"]} for v in report["models"].values()]
display(pd.DataFrame(rows).set_index("Mô hình"))
print("Được chọn theo MAE valid:", MODEL_LABELS[report["selected"]])
charts.model_comparison(report);

- Gradient Boosting sai ít nhất trên tập chọn nên được chọn **trước khi mở tập kiểm tra**; trên tập kiểm tra nó vẫn tốt nhất.
- Gradient Boosting đoán tốt hơn hẳn trên tập học so với tập chọn: với khoảng 1.200 tin để học, mô hình đã "học thuộc" một phần.
- Sai số cao hơn bản trước kia có lẫn tin do máy sinh (5,82 triệu), vì tin thật khó đoán hơn. Hai con số này không so trực tiếp được.

## 6. Mô hình đoán sai nhiều ở đâu?

In [ ]:
pred = pd.read_csv(artifacts_dir() / "test_predictions.csv")
error = (pred["predicted"] - pred["salary"]).abs() / pred["salary"]
print(f"Một nửa số tin test lệch dưới {error.median():.1%}; {(error <= 0.2).mean():.0%} lệch không quá 20%")
charts.actual_vs_predicted(pred);

In [ ]:
charts.error_by_role(pred);

Sai nhiều nhất ở Tech Lead/Architect và vị trí quản lý: cùng chức danh nhưng lương ở một startup 20 người khác xa một tập đoàn 5.000 người, mà dữ liệu không có thông tin quy mô công ty.

## 7. Yếu tố quyết định lương và kỹ năng đáng học
**Cách đo mức quan trọng:** xáo trộn một cột thông tin rồi xem mô hình đoán kém đi bao nhiêu. Giống như cho một cầu thủ nghỉ một trận để biết anh ấy quan trọng với đội thế nào.

In [ ]:
charts.feature_importance(report);

In [ ]:
value = load_skill_value()
display(value[["skill", "postings", "premium_median", "model_uplift"]].rename(columns={
    "skill": "Kỹ năng", "postings": "Số tin", "premium_median": "Chênh trung vị thô", "model_uplift": "Mức tăng mô hình"}).head(12))
print("Kỹ năng có mức tăng âm:", int((value["model_uplift"] < 0).sum()))
charts.skill_uplift(value);

⚠️ Mức tăng lương ở đây là phép thử "nếu như" trên một mô hình học từ tin tuyển dụng. Nó cho thấy hai thứ hay đi cùng nhau, **chưa chứng minh** học kỹ năng đó sẽ được tăng lương.